In [0]:
# --- CẤU HÌNH KẾT NỐI ---
storage_account_name = "banhangdata"
container_name = "dataphantich-2025-11-18t07-44-02-888z"
access_key = ""

mount_point = "/mnt/container"

if not any(mount.mountPoint == mount_point for mount in dbutils.fs.mounts()):
    try:
        dbutils.fs.mount(
            source = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net",
            mount_point = mount_point,
            extra_configs = {f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net": access_key}
        )
        print("Mount thành công!")
    except Exception as e:
        print("Lỗi mount:", e)
else:
    print("Đã mount trước đó rồi.")

Mount thành công!


In [0]:
import pandas as pd
import os

file_path = '/dbfs/mnt/container/raw_data/Online Retail.xlsx'

df = pd.read_excel(file_path, engine='openpyxl')
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB
None
            Quantity      UnitPrice     CustomerID
count  541909.000000  541909.000000  406829.000000
mean        9.552250       4.611114   15287.690570
std       218.081158      96.759853    1713.600303
min    -80995.000000  -11062.060000   12346.000000
25%         1.000000       1.250000   13953.0000

In [0]:
df_term = df.copy()
print(df_term.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB
None


In [0]:
df_term['CustomerID'] = pd.to_numeric(df_term['CustomerID'], errors='coerce').astype('Int64')

In [0]:
#Loại bỏ các dòng có mã hóa đơn không hợp lệ
import pandas as pd

df_term['InvoiceNo'] = pd.to_numeric(df_term['InvoiceNo'], errors='coerce')
df_term.dropna(subset=['InvoiceNo'], inplace=True)
df_term = df_term[(df_term['InvoiceNo'] >= 100000) & (df_term['InvoiceNo'] <= 999999)]
df_term['InvoiceNo'] = df_term['InvoiceNo'].astype(int)
df_term.dropna(inplace=True)

# Kiểm tra lại kết quả
print("\nKích thước của DataFrame cuối cùng:")
print(df_term.shape)


Kích thước của DataFrame cuối cùng:
(397924, 8)


In [0]:
# Lọc bỏ các dòng không phải sản phẩmphẩm
non_product_codes = ['POST', 'M', 'C2', 'DOT', 'BANK CHARGES', 'PADS']
df_term = df_term[~df_term['StockCode'].isin(non_product_codes)]
print(df_term.shape)

(396370, 8)


In [0]:
# Xóa duplicates
print(f"Số dòng trùng: {df_term.duplicated().sum()}")
df_term[df_term.duplicated(keep=False)].head(50)
df_term = df_term.drop_duplicates()
print(df_term.shape)

Số dòng trùng: 5187
(391183, 8)


In [0]:
# Sanity Check (Kiểm tra tính hợp lý)
import pandas as pd
import os

output_dir = "column_values"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print("Bắt đầu xử lý các cột...")

for column in df_term.columns:
    print(f"\n--- Đang xử lý cột: {column} ---")

    if pd.api.types.is_numeric_dtype(df_term[column]):

        negative_values = df_term[column][df_term[column] < 0].unique()

        if len(negative_values) > 0:
            print(f"(!) Tìm thấy các giá trị âm trong cột '{column}':")
            print(negative_values)
        else:
            print(f"(i) Cột '{column}' không có giá trị âm.")

    value_counts_sorted = df_term[column].value_counts()

    file_path = os.path.join(output_dir, f"frequency_{column}.txt")

    try:
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(f"Danh sách giá trị trong cột '{column}', sắp xếp theo tần suất giảm dần:\n")
            f.write("------------------------------------------------------------------\n")
            for value, count in value_counts_sorted.items():
                f.write(f"{value} (xuất hiện {count} lần)\n")

        print(f"-> Đã tạo thành công file: {file_path}")

    except Exception as e:
        print(f"(x) Lỗi khi ghi file cho cột '{column}': {e}")

print("\nHoàn tất quá trình!")

Bắt đầu xử lý các cột...

--- Đang xử lý cột: InvoiceNo ---
(i) Cột 'InvoiceNo' không có giá trị âm.
-> Đã tạo thành công file: column_values/frequency_InvoiceNo.txt

--- Đang xử lý cột: StockCode ---
-> Đã tạo thành công file: column_values/frequency_StockCode.txt

--- Đang xử lý cột: Description ---
-> Đã tạo thành công file: column_values/frequency_Description.txt

--- Đang xử lý cột: Quantity ---
(i) Cột 'Quantity' không có giá trị âm.
-> Đã tạo thành công file: column_values/frequency_Quantity.txt

--- Đang xử lý cột: InvoiceDate ---
-> Đã tạo thành công file: column_values/frequency_InvoiceDate.txt

--- Đang xử lý cột: UnitPrice ---
(i) Cột 'UnitPrice' không có giá trị âm.
-> Đã tạo thành công file: column_values/frequency_UnitPrice.txt

--- Đang xử lý cột: CustomerID ---
(i) Cột 'CustomerID' không có giá trị âm.
-> Đã tạo thành công file: column_values/frequency_CustomerID.txt

--- Đang xử lý cột: Country ---
-> Đã tạo thành công file: column_values/frequency_Country.txt

Hoàn t

In [0]:
print(df_term.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 391183 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    391183 non-null  int64         
 1   StockCode    391183 non-null  object        
 2   Description  391183 non-null  object        
 3   Quantity     391183 non-null  int64         
 4   InvoiceDate  391183 non-null  datetime64[ns]
 5   UnitPrice    391183 non-null  float64       
 6   CustomerID   391183 non-null  Int64         
 7   Country      391183 non-null  object        
dtypes: Int64(1), datetime64[ns](1), float64(1), int64(2), object(3)
memory usage: 27.2+ MB
None


In [0]:
# Lưu file CSV
df_term.to_csv('/dbfs/mnt/container/input_data/input.csv', index=False, encoding='utf-8')